# Exploratory Notebook
## Introduction
We are going to build a RAG system for the French company Calogena. In this notebook, we will:
- examine the publicly available material regarding the company
- check if the extracted texts are clean and make corrections if necessary
- determine the chunk size for the RAG system
- select the appropriate embedding model
- test our assistant

In [10]:
from pathlib import Path
import urllib.request
from pypdf import PdfReader
import pandas as pd 
from collections import Counter 
import re 

## 01. Data Collection

To begin with, we will look for the documents needed for our RAG system.            

We require reliable documents containing verified information. We also need a substantial amount of content for our embedding process to work effectively. We will therefore turn to the ASN, the Nuclear Safety Authority, which has published sourced documents, each around ten pages long regarding Calogena.

Here is the first PDF, the 2026 ASNR expert assessment: It evaluates the CAL-30, the reactor Calogena intends to use, which features two emergency shutdown systems. The document focuses on these systems.       
[Avis d'expertise ASNR 2026](https://recherche-expertise.asnr.fr/sites/default/files/2026-06/ASNR-avis-expertise-2026-00015.pdf)

The second is the 2026 Board opinion: This represents the official position based on the expert assessment. It addresses the same emergency shutdown systems, deeming them satisfactory at this stage, while calling for a simple, robust alternative method.    
[Avis ASNR 2026](https://reglementation-controle.asnr.fr/content/download/209769/file/2026-AV-027.pdf)   

Both of these PDFs are relatively short, about ten pages each. We will look at the implications of this later.

So we are going to download them.       
[Python Documentation on urllib](https://docs.python.org/3/library/urllib.request.html)

In [3]:
raw_path = Path("data/raw")

# 2026 expert assessment
urllib.request.urlretrieve(
    "https://recherche-expertise.asnr.fr/sites/default/files/2026-06/ASNR-avis-expertise-2026-00015.pdf",
    raw_path / "avis_expertise_ASNR_2026.pdf"
)

# 2026 Board opinion
urllib.request.urlretrieve(
    "https://reglementation-controle.asnr.fr/content/download/209769/file/2026-AV-027.pdf",
    raw_path / "avis_ASNR_2026.pdf"
)

(WindowsPath('data/raw/avis_ASNR_2026.pdf'),
 <http.client.HTTPMessage at 0x16f61d44810>)

We will now read these PDFs         
[pypdf documentation on text extraction](https://pypdf.readthedocs.io/en/stable/user/extract-text.html)

We will use the example from the documentation but take all the pages and store them in a list. For each page, this list will contain a dictionary with the filename, the page number, and the text. We created a function to do it so we could reuse it.

In [4]:
def register_pdf(folder : Path)-> list | None: 
    """
    Reads all PDFs in the folder and returns a list where each page is a dictionary 
    containing the original filename, the page number, and the text.
    folder : path
    """
    pages = []
    for path in sorted(folder.glob("*.pdf")):
        try:
            reader = PdfReader(path)
            for i, page in enumerate(reader.pages, start=1): # enumerate take the number and the page
                pages.append({"origin": path.name, "page": i, "text": page.extract_text() or ""})
        except Exception as e:
            print(f"Error for {path.name} : {e}")
    return pages 

Now that we have created this function, we can convert it into a table. Each dictionary will become a row in the table.

In [5]:
df_pages = pd.DataFrame(register_pdf(raw_path))

print(df_pages.head(1))

               origin  page                                               text
0  avis_ASNR_2026.pdf     1  RÉPUBLIQUE FRANÇAISE \n \n \n \n 1/4 \n \n \nA...


It worked, we can clearly see the file name, the page number, and the beginning of the text. Pandas truncates the text because it is too long.

We will therefore check the text by displaying a full page.

In [6]:
print(df_pages.loc[0, "text"][:5000])

RÉPUBLIQUE FRANÇAISE 
 
 
 
 1/4 
 
 
Avis n° 2026-AV-027 de l’Autorité de sûreté nucléaire et de radioprotection  
du 19 mai 2026 relatif aux options de sûreté présentées par CALOGENA 
concernant sa démarche de diversification du système d’arrêt du réacteur CAL-30 
 
 
L’Autorité de sûreté nucléaire et de radioprotection,  
Vu le code de l’environnement, notamment le titre IX de son livre V et son article R. 593-14 ;   
Vu le code de la santé publique, notamment son article L. 1333-2 ;  
Vu l’arrêté du 7 février 2012 modifié fixant les règles générales relatives aux installations nucléaires de base ;  
Vu la décision n° 2015-DC-0532 de l’Autorité de sûreté nucléaire du 17 novembre 2015 relative au rapport de 
sûreté des installations nucléaires de base ; 
Vu le guide de l’Autorité de sûreté nucléaire n° 22 du 18 juillet 2017 relatif à la conception des réacteurs à eau 
sous pression ; 
Vu le courrier de CALOGENA référenc é CAL-030-L-0-GEN-LEE-001-01 du 31 octobre 2024, modifié par le 

We can therefore observe:
- line breaks in the middle of sentences
- words split in two
- double spaces
- "Vu..." referring to legal citations, which can clutter the explanations regarding the reactor

We therefore need to:
- replace line breaks with spaces
- remove double spaces
- ignore split words, as this is a minor issue

Now, let us consider how to reconstruct the broken lines while preserving the paragraphs. We need to compare more pages, let's look at some others.

In [7]:
print(df_pages.loc[5, "text"][:5000])

 
  
AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026 
  
2/6 
 
Le refroidissement du cœur repose sur la circulation naturelle de l’eau primaire pour toutes les situations du 
réacteur (fonctionnement normal, incidents et accidents) : l’eau chauffée par le cœur s’élève dans une colonne de 
grande hauteur , appelée cheminée centrale , puis redescend au travers de  l’échangeur primaire/secondaire 
implanté autour de la cheminée pour retourner dans le cœur. 
1.2. SYSTÈMES DE MAÎTRISE DE LA RÉACTIVITÉ EN SITUATIONS INCIDENTELLES ET ACCIDENTELLES 
La maîtrise de la réactivité du réacteur CAL -30 en situations incidentelles et accidentelles repose sur l’insertion 
automatique de grappes de crayons d’absorbants neutroniques dans le cœur, ces dernières étant réparties en un 
groupe de régulation et deux groupes d’arrêt. Le réacteur est équipé de deux systèmes d’arrêt d’urgence (AU), le 
premier, appelé « système d’AU principal », provoquant la chute gravitaire d es grappes du groupe de régul

In [9]:
print(df_pages.loc[8, "text"][:5000])

 
  
AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026 
  
5/6 
 
2.2.4. Enjeux associés à la défaillance de chute de grappes par mode commun 
Afin d’apprécier les enjeux de sûreté, la Direction de l’expertise en sûreté a demandé à la société CALOGENA, 
au cours de l’expertise, de fournir des éléments permettant d’apprécier les conséquences de la défaillance de la 
chute de toutes les grappes. La société CALOGENA a alors transmis des éléments visant à justifier  l’existence 
d’un délai significatif avant que le refroidissement du cœur du réacteur ne soit compromis. Dans le cas où l’échec 
de la chute des grappes proviendrait de défaillances d es systèmes de contrôle-commande ou des dispositifs de 
coupure de l’alimentation électrique des mécanismes, elle a alors précisé que des actions en local pourraient être 
menées dans ce délai en vue de  provoquer la chute des grappes par coupure forcée de l’alimentation électrique 
des mécanismes. L’ensemble de ces éléments préliminaires devra êt

We can observe that:
- the header is repeated on every page
- semicolons mark the end of legal paragraphs
- paragraphs often end with a period followed by more than three spaces

We will test our hypothesis regarding paragraphs marked by a period followed by more than three spaces across all documents.

In [13]:
counter = Counter() 
for text in df_pages["text"]:
    for line in text.split("\n"):
        paragraph = re.search(r"\. {3,}$", line)
        if paragraph:
            counter["3 spaces or more"] += 1 
counter

Counter({'3 spaces or more': 6})

There are only 6 examples of this pattern, a period followed by at least three spaces, so that is too few.

Instead, we will use pypdf's layout mode. This way, we will have the vertical spacing that marks the paragraphs.
Let us therefore evolve our function to incorporate this mode.

In [ ]:
def register_pdf(folder : Path, mode : str)-> list | None: 
    """
    Reads all PDFs in the folder and returns a list where each page is a dictionary 
    containing the original filename, the page number, and the text.
    folder : path
    """
    pages = []
    for path in sorted(folder.glob("*.pdf")):
        try:
            reader = PdfReader(path)
            for i, page in enumerate(reader.pages, start=1): # enumerate take the number and the page
                text = page.extract_text(extraction_mode=mode)
                pages.append({"origin": path.name, "page": i, "text": text or ""})
        except Exception as e:
            print(f"Error for {path.name} : {e}")
    return pages 

In [24]:
df_pages = pd.DataFrame(register_pdf(raw_path, mode="layout"))
print(df_pages.head(1))
print(df_pages.loc[0, "text"][:5000])

               origin  page                                               text
0  avis_ASNR_2026.pdf     1                                                ...
                                                                            RÉPUBLIQUE FRANÇAISE

















         Avis n° 2026-                                                 AV-027                                                         de l’Autorité de sûreté nucléaire et de radioprotection
            du 19 mai 2026  relatif  aux options  de sûreté  présentées par CALOGENA
concernant sa démarche  de  diversification  du  système d’arrêt du  réacteur CAL-30



L’Autorité de s ûreté nucléaire                     et de radiopr otection,

Vu le code de l’env ironnement, notamment le titre IX de son livre V et son article R. 593-                    14 ;

Vu le code de la santé publique, notamment son article L. 1333-                                                          2 ;

Vu l’arrêté du 7 février 2012 modifié fixant 

In [25]:
print(df_pages.loc[5, "text"][:5000])

Le  refroidissement  du  cœur  repose  sur  la  circulation  naturelle  de  l’eau  primaire  pour  toutes  les  situations  du
réacteur (fonctionnement normal, incidents et accidents) : l’eau chauffée par le cœur s’élève dans une colonne de
grande  hauteur,     appelée     cheminée  centrale,       puis  redescend      au  travers  de    l’échangeur  primaire/secondaire
implanté autour de la cheminée pour retourner dans le cœur.
1.2.     SYSTÈMES DE MAÎTRISE DE LA RÉACTIVITÉ EN SITUATIONS INCIDENTELLES ET ACCIDENTELLES
La maîtrise de la réactivité du réacteur CAL-30 en situations incidentelles et accidentelles                     repose sur l’insertion
automatique de grappes de crayons d’absorbants neutroniques dans le cœur, ces dernières étant réparties en un
groupe de régulation et deux groupes d’arrêt. Le réacteur est équipé de deux systèmes d’arrêt d’urgence (AU), le
premier, appelé « système d’AU principal », provoquant la chute gravitaire des grappes du groupe de régulation
et de

The two documents do not operate in the same way. The "layout" approach works better for the expert assessment, whereas the "plain text" format works better for the board's opinion. We will therefore read these two documents differently.

Let's update the function to select a single element, and take the opportunity to set a default mode.

In [34]:
def register_pdf(path : Path, mode : str = "plain")-> list | None: 
    """
    Reads all PDFs in the folder and returns a list where each page is a dictionary 
    containing the original filename, the page number, and the text.
    folder : path
    """
    pages = []
    try:
        reader = PdfReader(path)
        for i, page in enumerate(reader.pages, start=1): # enumerate take the number and the page
            text = page.extract_text(extraction_mode=mode)
            pages.append({"origin": path.name, "page": i, "text": text or ""})
    except Exception as e:
        print(f"Error for {path.name} : {e}")
    return pages 

In [35]:
pages = (
    register_pdf(raw_path / "avis_expertise_ASNR_2026.pdf", mode="layout") +
    register_pdf(raw_path / "avis_ASNR_2026.pdf", mode="plain")
)

df_pages = pd.DataFrame(pages)
print(df_pages.loc[0, "text"][:5000])

                                                                  Monsieur le Directeur de la Direction des réacteurs innovants


                                                                                                Fontenay-aux-Roses, le 2 février 2026

           AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026

Objet :                CALOGENA – Projet de réacteur CAL-30 - Démarche de diversification pour les systèmes
                       assurant   la   maîtrise   de   la   réactivité   du   réacteur   en   situations   incidentelles   et
                       accidentelles.
Références :           [1] Saisine ASNR - CODEP-DRI-2025-028912 du 5 mai 2025.
                       [2] Arrêté du 7 février 2012 fixant les règles générales relatives aux installations nucléaires de
                           base.
                       [3] Guide ASN n° 22 : Conception des réacteurs à eau sous pression.

Dans  le  cadre  du  développement  de  son               projet  de  réact

In [37]:
print(df_pages.loc[5, "text"][:5000])

                                                                 ANNEXE
                            Recommandation de la Direction de l’expertise en sûreté



La Direction de l’expertise en sûreté recommande que, afin d’assurer, avec un haut niveau de fiabilité, la fonction
de  sûreté  liée  à  la  maîtrise  de  la  réactivité,  la  société  CALOGENA  prévoie              un  moyen  complémentaire  aux
dispositions prévues pour la fonction d’arrêt d’urgence, qui soit fonctionnellement diversifié.
































































































                                       AVIS D’EXPERTISE N° 2026-00015 DU 02 FÉVRIER 2026                                                6/6




Right, we have a clearer picture of the issues we still need to resolve:
- excessive spacing
- removing headers and footers
- removing multiple blank lines
- joining lines belonging to the same paragraph
- splitting paragraphs